## Akatsen interpretazioa: negatibo eta positibo faltsuak

- **Negatibo faltsua (FN):** makinak huts egiten du, baina ereduak ez duela huts egingo aurreikusten du. Akatsik larriena da, mantentze-lana ez delako programatzen eta geldialdiak 1.200 €/orduko kostua izan dezakeelako.
- **Positibo faltsua (FP):** ereduak hutsegite baten berri ematen du, baina azkenean hutsegiterik ez da gertatzen. Teknikariaren berrikuspenak 210 € balio du.

Horregatik doitzen da atalasea Recall lehenesteko, positibo faltsuen igoera kontrolatua onartuz. Azken erabakia mantentze mota bakoitzaren benetako kostuarekin batera berrikusi behar da.

# CNC Guard — Entrenamendua eta ebaluazioa

Notebook honek CSVaren garbiketa, klaseen desoreka, feature engineering-a, Recall-a lehenesten duten metriken bidezko ebaluazioa eta `scoring-api`-k erabiltzen duen artefaktuaren gordetzea dokumentatzen ditu.

Negozio-helburua negatibo faltsuak murriztea da: aurreikusi gabeko geldialdi batek 1.200 € balio du eta alerta bakoitzeko teknikariaren berrikuspena 210 €. Atalasea kostu ekonomiko osoa minimizatuz aukeratuko da, gutxienez %76ko Recall mantenduta.

## 1. Datuak prestatzea eta kargatzea

`ai4i2020_train.csv` soilik erabiltzen da; simulagailuaren datuak ez dira entrenamenduarekin nahasten.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scoring'))
from train_model import (
    DATA_PATH, ENGINEERED_FEATURE_COLUMNS, FEATURE_COLUMNS,
    QUALITY_COLUMN, TARGET_COLUMN, add_engineered_features,
    clean_training_data, choose_failure_threshold,
)

raw = pd.read_csv(ROOT / 'data' / 'ai4i2020_train.csv')
df = clean_training_data(raw)
print(f'Hasierako errenkadak: {len(raw):,}')
print(f'Garbiketaren ondorengo errenkadak: {len(df):,}')
df.head()

## 2. EDA eta klaseen desoreka

Hutsegiteen klasea gutxiengoa da. Horregatik, Accuracy ez da nahikoa: beti 0 aurreikusten duen eredu batek accuracy altua izango luke, baina Recall 0 izango litzateke.

In [ ]:
display(df.describe(include='all').T)
print('machine_type banaketa:')
display(df[QUALITY_COLUMN].value_counts())
print('Helburuaren banaketa:')
display(df[TARGET_COLUMN].value_counts())
print(f'Hutsegite-tasa: {df[TARGET_COLUMN].mean():.2%}')

df[TARGET_COLUMN].value_counts().sort_index().plot(kind='bar', color=['#4caf50', '#e53935'])
plt.title('Hutsegiteen banaketa')
plt.xlabel('machine_failure')
plt.ylabel('Erregistro kopurua')
plt.show()

## 3. Garbiketa eta feature engineering-a

`temp_diff` aldagaiak prozesuaren eta giroaren tenperaturen arteko aldea adierazten du. `power_est` aldagaiak karga mekanikoa hurbiltzen du, `rpm × torque_Nm` formularen bidez. Aldagai hauek interpretazio fisikoa dute eta APIan modu berean kalkulatzen dira inferentzian.

In [ ]:
df = add_engineered_features(df)
display(df[[*FEATURE_COLUMNS, *ENGINEERED_FEATURE_COLUMNS, TARGET_COLUMN]].head())
display(df[[*ENGINEERED_FEATURE_COLUMNS, TARGET_COLUMN]].groupby(TARGET_COLUMN).mean())

## 4. Zatiketa data leakage-rik gabe

Zatiketa estratifikatua da hutsegiteen proportzioa mantentzeko. Ereduak ez ditu balidazio-multzoko datuak ikusten entrenamenduan.

In [ ]:
from sklearn.model_selection import train_test_split

NUMERIC_FEATURES = [*FEATURE_COLUMNS, *ENGINEERED_FEATURE_COLUMNS]
MODEL_FEATURES = [*NUMERIC_FEATURES, QUALITY_COLUMN]
X = df[MODEL_FEATURES]
y = df[TARGET_COLUMN].astype(int)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_development, y_development, test_size=0.25, random_state=42, stratify=y_development
)
print(f'Treinamendua: {len(X_train):,}; balidazioa: {len(X_val):,}; testa: {len(X_test):,} errenkada')
print(f'Hutsegiteak: entrenamenduan {y_train.mean():.2%}; balidazioan {y_val.mean():.2%}; testean {y_test.mean():.2%}')

## 5. Ereduen konparazioa

Random Forest gainbegiratua eta erregresio logistiko orekatua konparatzen dira. Ekoizpenerako aukeratutako eredua Random Forest da, sentsoreen arteko erlazio ez-linealak harrapatzen dituelako eta probabilitateak lortzeko aukera ematen duelako. `class_weight='balanced'` parametroak garrantzi handiagoa ematen die hutsegiteen klaseko adibideei.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def preprocessor():
    return ColumnTransformer([
        ('numeric', 'passthrough', NUMERIC_FEATURES),
        ('quality', OneHotEncoder(handle_unknown='ignore'), [QUALITY_COLUMN]),
    ])

models = {
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor()),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ]),
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor()),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)),
    ]),
}

comparison = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = probabilities >= 0.5
    comparison.append({
        'eredua': name,
        'ROC-AUC': roc_auc_score(y_val, probabilities),
        'Recall': recall_score(y_val, predictions, zero_division=0),
        'Precision': precision_score(y_val, predictions, zero_division=0),
        'F1': f1_score(y_val, predictions, zero_division=0),
    })
pd.DataFrame(comparison).sort_values('Recall', ascending=False)

## 6. Kostu ekonomikoaren araberako atalasea eta nahasmen-matrizea

210 €-ko berrikuspenaren eta detektatu gabeko hutsegitearen 1.200 €-ko kostuaren arabera, 0,76ko Recall minimoa betetzen duten atalaseen artean kostu oso txikiena hautatzen da. Balidazioan hautatu ondoren, emaitza test independentean egiaztatzen da.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

rf = models['Random Forest']
rf_probabilities = rf.predict_proba(X_val)[:, 1]
threshold, threshold_info = choose_failure_threshold(y_val, rf_probabilities, target_recall=0.76)
rf_predictions = rf_probabilities >= threshold
print(f'Hautatutako atalasea: {threshold:.4f}')
print(threshold_info)
print('Balidazioaren classification report:')
print(classification_report(y_val, rf_predictions, zero_division=0))

rf_test_probabilities = rf.predict_proba(X_test)[:, 1]
rf_test_predictions = rf_test_probabilities >= threshold
test_matrix = confusion_matrix(y_test, rf_test_predictions, labels=[0, 1])
tn, fp, fn, tp = test_matrix.ravel()
test_cost = (fp + tp) * 210 + fn * 1200
print('Test independentea:')
print(classification_report(y_test, rf_test_predictions, zero_division=0))
print(f'Testeko kostu osoa: {test_cost:,.0f} €')

ConfusionMatrixDisplay(
    confusion_matrix(y_val, rf_predictions),
    display_labels=['Hutsegiterik ez', 'Hutsegitea'],
).plot(cmap='Blues')
plt.title('Nahasmen-matrizea — Random Forest atalase doituarekin')
plt.show()

## 7. Aldagaien garrantzia

Garrantzia metriken batera interpretatzen da: ez du kausalitatea frogatzen, baina ereduak zein seinale erabiltzen dituen azaltzen laguntzen du.

In [ ]:
classifier = rf.named_steps['classifier']
pre = rf.named_steps['preprocessor']
names = [*NUMERIC_FEATURES, *pre.named_transformers_['quality'].get_feature_names_out([QUALITY_COLUMN])]
importance = pd.Series(classifier.feature_importances_, index=names).sort_values()
importance.plot(kind='barh', figsize=(8, 5), color='#1976d2')
plt.title('Aldagaien garrantzia — Random Forest')
plt.xlabel('Garrantzia')
plt.show()
importance.sort_values(ascending=False)

## 8. Azken entrenamendua eta gordetzea

Ekoizpeneko scriptak hutsegite moten sailkatzaileak eta Isolation Forest ere entrenatzen ditu, aukeratutako atalasea gordetzen du eta `models/cnc_failure_model.pkl` sortzen du.

In [ ]:
from train_model import main as train_production_artifact
train_production_artifact()
print('Artefaktua hemen gorde da:', ROOT / 'models' / 'cnc_failure_model.pkl')